# 🚀 Master Crypto Lookback Horizon & Regime Transfer Benchmark (Google Colab)
### Autonomous Quantitative Study: Training Horizons (6.0Y, 5.0Y, 4.0Y, 3.0Y, 2.75Y, 2.5Y, 2.25Y, 2.0Y)
**Assets:** `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `DOTUSDT` (1-Hour & 4-Hour Candles)  
**Execution Architecture:** 20-bar Donchian Breakouts + 200 SMA Trend Filter + LightGBM Meta-Labeling  
**Author:** Anshul Parate & Antigravity Quantitative Research Team

---

### Key Experimental Objectives:
1. **Multi-Horizon Lookback Sweep:**
   Train dedicated LightGBM meta-labeling models on distinct training horizons ending on `2026-10-21 00:00:00 UTC`:
   - **6.0 Years** (Oct 2020 – Oct 2026)
   - **5.0 Years** (Oct 2021 – Oct 2026)
   - **4.0 Years** (Oct 2022 – Oct 2026)
   - **3.0 Years** (Oct 2023 – Oct 2026)
   - **2.75 Years** (Jan 2024 – Oct 2026) - *Post-ETF modern institutional regime*
   - **2.50 Years** (Apr 2024 – Oct 2026)
   - **2.25 Years** (Jul 2024 – Oct 2026) - *Commodity-aligned window*
   - **2.00 Years** (Oct 2024 – Oct 2026)

2. **Dual-Directional Out-of-Sample Evaluation:**
   - **Forward Live Evaluation (Oct 21 – Nov 7, 2026):**
     Tests how well each model generalizes to real-time unseen live feeds.
   - **Backward Regime Transfer (Historical Stress-Test on "Leftover" Earlier Data):**
     When a model is trained on a modern window (e.g. 2.75 years), we evaluate its predictions on the **excluded earlier historical data (2020–2023)** to measure how well modern decision trees survive previous market regimes (2021 bull market, 2022 Terra/FTX bear market).

3. **Autonomous Export:**
   Automatically selects the best model per asset, converts to TypeScript-compatible LightGBM JSON trees, and packages into a `.zip` ready for dashboard deployment.


In [ ]:
# 1. Environment Setup & Dependency Installation
import os
import sys
import json
import math
import time
import zipfile
import urllib.request
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any, Tuple, Optional

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print(f"NumPy Version   : {np.__version__}")
print(f"Pandas Version  : {pd.__version__}")
print(f"LightGBM Version: {lgb.__version__}")
print("✓ Environment initialized successfully.")


## Step 1: Binance Historical Ingestion

In [ ]:
# 2. Autonomous Binance Futures Historical Data Ingestion (2020 - 2026)
# Fetches complete historical 1H candles directly from Binance Public API

CRYPTOS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "DOTUSDT"]

def fetch_binance_klines(symbol: str, interval: str = "1h", start_year: int = 2020) -> pd.DataFrame:
    print(f"📥 Fetching {symbol} {interval} klines from Binance API since {start_year}...")
    start_ts = int(datetime(start_year, 1, 1).timestamp() * 1000)
    end_ts = int(datetime(2026, 11, 8).timestamp() * 1000)
    
    all_bars = []
    current_end = end_ts
    
    while current_end > start_ts:
        url = f"https://fapi.binance.com/fapi/v1/klines?symbol={symbol}&interval={interval}&limit=1000&endTime={current_end}"
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "TraderPro-Colab-Research/2.0"})
            with urllib.request.urlopen(req, timeout=15) as resp:
                data = json.loads(resp.read().decode())
            if not data:
                break
            all_bars = data + all_bars
            current_end = data[0][0] - 1
            if len(data) < 1000 or current_end <= start_ts:
                break
            time.sleep(0.04) # API courteous backoff
        except Exception as e:
            print(f"  Warning on {symbol}: {e}. Retrying in 1s...")
            time.sleep(1.0)
            
    if not all_bars:
        raise RuntimeError(f"Could not fetch Binance data for {symbol}")
        
    cols = ["ot", "open", "high", "low", "close", "volume", "ct", "qv", "n", "tbv", "tbqv", "i"]
    df = pd.DataFrame(all_bars, columns=cols)
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = df[c].astype(float)
    df["open_time"] = pd.to_datetime(df["ot"], unit="ms", utc=True)
    df = df.drop_duplicates("open_time").sort_values("open_time").reset_index(drop=True)
    
    print(f"  ✓ Loaded {len(df):>5} candles ({df['open_time'].min().strftime('%Y-%m-%d')} to {df['open_time'].max().strftime('%Y-%m-%d')})")
    return df

# Ingest all 4 crypto assets (1H)
raw_dfs_1h = {}
raw_dfs_4h = {}

for sym in CRYPTOS:
    df1 = fetch_binance_klines(sym, interval="1h", start_year=2020)
    raw_dfs_1h[sym] = df1
    
    # Resample to 4H candles
    df4 = df1.copy().set_index("open_time").resample("4h").agg({
        "open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"
    }).dropna().reset_index()
    raw_dfs_4h[sym] = df4

print("\n✓ All crypto klines ingested and resampled to 4H successfully.")


## Step 2: Feature Engineering & Triple Barrier Meta-Labeling

In [ ]:
# 3. Universal 10-Feature Schema & Triple-Barrier Labeling

FEATURE_NAMES = [
    "feat_atr_pct",
    "feat_vol_zscore",
    "feat_volatility_ratio",
    "feat_donchian_pos",
    "feat_trend_dist_200",
    "feat_mom_24h",
    "feat_mom_72h",
    "feat_rsi_14",
    "feat_dist_ema20",
    "feat_dist_ema50",
]

CRYPTO_FRICTION_PCT = 0.00118  # 11.8 bps taker fee + spread + GST

def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    c, h, l, v = df["close"], df["high"], df["low"], df["volume"]

    # True Range & ATR
    prev_c = c.shift(1)
    tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    atr14 = tr.rolling(14, min_periods=14).mean()
    atr7 = tr.rolling(7, min_periods=7).mean()
    atr28 = tr.rolling(28, min_periods=28).mean()
    df["atr_14"] = atr14

    df["feat_atr_pct"] = atr14 / (c + 1e-8)
    v_mean20 = v.rolling(20, min_periods=5).mean()
    v_std20 = v.rolling(20, min_periods=5).std()
    df["feat_vol_zscore"] = (v - v_mean20) / (v_std20 + 1e-8)
    df["feat_volatility_ratio"] = atr7 / (atr28 + 1e-8)

    donch_high20 = h.shift(1).rolling(20, min_periods=20).max()
    donch_low20 = l.shift(1).rolling(20, min_periods=20).min()
    df["donch_high20"] = donch_high20
    df["donch_low20"] = donch_low20
    channel_width = donch_high20 - donch_low20
    df["feat_donchian_pos"] = (c - donch_low20) / (channel_width + 1e-8)

    sma200 = c.rolling(200, min_periods=30).mean()
    df["sma200"] = sma200
    df["feat_trend_dist_200"] = (c - sma200) / (sma200 + 1e-8)

    df["feat_mom_24h"] = (c - c.shift(24)) / (c.shift(24) + 1e-8)
    df["feat_mom_72h"] = (c - c.shift(72)) / (c.shift(72) + 1e-8)

    delta = c.diff()
    gain = delta.clip(lower=0).rolling(14, min_periods=14).mean()
    loss = (-delta.clip(upper=0)).rolling(14, min_periods=14).mean()
    rs = gain / (loss + 1e-8)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    df["feat_rsi_14"] = (rsi - 50.0) / 50.0

    df["feat_dist_ema20"] = (c - c.ewm(span=20, adjust=False).mean()) / (c + 1e-8)
    df["feat_dist_ema50"] = (c - c.ewm(span=50, adjust=False).mean()) / (c + 1e-8)

    for col in FEATURE_NAMES:
        df[col] = df[col].fillna(0.0).clip(-5.0, 5.0)

    return df

def generate_triple_barrier_setups(df: pd.DataFrame, timeframe: str = "1h") -> List[Dict[str, Any]]:
    trades = []
    min_stop, max_stop = 0.015, 0.060
    max_hold_bars = 16 if timeframe == "1h" else 12
    n = len(df)
    last_exit_idx = -1

    for i in range(200, n - max_hold_bars):
        if i <= last_exit_idx:
            continue
        c = df["close"].iloc[i]
        d_high = df["donch_high20"].iloc[i]
        d_low = df["donch_low20"].iloc[i]
        sma = df["sma200"].iloc[i]

        if math.isnan(d_high) or math.isnan(d_low) or math.isnan(sma):
            continue

        side = None
        if c > d_high and c > sma:
            side = "LONG"
        elif c < d_low and c < sma:
            side = "SHORT"
        if not side:
            continue

        if side == "LONG":
            raw_stop = (c - d_low) / c
            stop_pct = max(min_stop, min(max_stop, raw_stop))
            sl = c * (1.0 - stop_pct)
            tp = c * (1.0 + 2.5 * stop_pct)
        else:
            raw_stop = (d_high - c) / c
            stop_pct = max(min_stop, min(max_stop, raw_stop))
            sl = c * (1.0 + stop_pct)
            tp = c * (1.0 - 2.5 * stop_pct)

        gross_r = 0.0
        exit_bar = i + max_hold_bars
        exit_price = df["close"].iloc[exit_bar]
        exit_type = "TIME"

        for offset in range(1, max_hold_bars + 1):
            curr = i + offset
            bh, bl = df["high"].iloc[curr], df["low"].iloc[curr]
            if side == "LONG":
                hit_tp, hit_sl = bh >= tp, bl <= sl
                if hit_tp and hit_sl:
                    gross_r, exit_type, exit_bar, exit_price = -1.0, "COLLISION", curr, sl
                    break
                elif hit_tp:
                    gross_r, exit_type, exit_bar, exit_price = 2.5, "TARGET", curr, tp
                    break
                elif hit_sl:
                    gross_r, exit_type, exit_bar, exit_price = -1.0, "STOP", curr, sl
                    break
            else:
                hit_tp, hit_sl = bl <= tp, bh >= sl
                if hit_tp and hit_sl:
                    gross_r, exit_type, exit_bar, exit_price = -1.0, "COLLISION", curr, sl
                    break
                elif hit_tp:
                    gross_r, exit_type, exit_bar, exit_price = 2.5, "TARGET", curr, tp
                    break
                elif hit_sl:
                    gross_r, exit_type, exit_bar, exit_price = -1.0, "STOP", curr, sl
                    break

        if exit_type == "TIME":
            gross_r = (exit_price - c) / (c * stop_pct) if side == "LONG" else (c - exit_price) / (c * stop_pct)

        cost_r = CRYPTO_FRICTION_PCT / stop_pct
        net_r = gross_r - cost_r
        y = 1 if gross_r >= 1.5 else 0

        trades.append({
            "entry_time": df["open_time"].iloc[i],
            "side": side,
            "entry_price": c,
            "exit_price": exit_price,
            "stop_pct": stop_pct,
            "gross_r": gross_r,
            "cost_r": cost_r,
            "net_r": net_r,
            "y": y,
            "features": [float(df[col].iloc[i]) for col in FEATURE_NAMES]
        })
        last_exit_idx = exit_bar

    return trades

print("✓ Universal 10-Feature schema and Triple Barrier engine ready.")


## Step 3: Multi-Horizon Lookback Sweep & Dual Evaluation (Forward Live + Backward Past Transfer)

In [ ]:
# 4. Master Lookback Horizon Sweep & Dual-Directional Evaluation Engine

LOOKBACK_HORIZONS = [
    {"label": "6.0 Years", "years": 6.00},
    {"label": "5.0 Years", "years": 5.00},
    {"label": "4.0 Years", "years": 4.00},
    {"label": "3.0 Years", "years": 3.00},
    {"label": "2.75 Years (Modern)", "years": 2.75},
    {"label": "2.50 Years", "years": 2.50},
    {"label": "2.25 Years (Comm-Align)", "years": 2.25},
    {"label": "2.00 Years", "years": 2.00},
]

IN_SAMPLE_CUTOFF = pd.Timestamp("2026-10-21 00:00:00", tz="UTC")
LIVE_END_CUTOFF = pd.Timestamp("2026-11-08 00:00:00", tz="UTC")

def run_lookback_regime_study(timeframe: str = "1h") -> pd.DataFrame:
    print(f"==========================================================================")
    print(f"🚀 EXECUTING MULTI-HORIZON STUDY FOR [{timeframe.upper()}] CRYPTO TIMEFRAME")
    print(f"==========================================================================")
    
    dfs_dict = raw_dfs_1h if timeframe == "1h" else raw_dfs_4h
    results = []
    
    for sym in CRYPTOS:
        print(f"\n🔍 Processing {sym}...")
        df_feat = compute_features(dfs_dict[sym])
        all_setups = generate_triple_barrier_setups(df_feat, timeframe=timeframe)
        
        # 1. Segregate live unseen window (Oct 21 - Nov 7, 2026)
        live_unseen = [t for t in all_setups if IN_SAMPLE_CUTOFF <= t["entry_time"] <= LIVE_END_CUTOFF]
        all_historical = [t for t in all_setups if t["entry_time"] < IN_SAMPLE_CUTOFF]
        
        for horizon in LOOKBACK_HORIZONS:
            h_years = horizon["years"]
            h_label = horizon["label"]
            
            # Compute start cutoff for this training horizon
            days_back = int(h_years * 365.25)
            start_cutoff = IN_SAMPLE_CUTOFF - pd.Timedelta(days=days_back)
            
            # Partition in-sample vs leftover past data
            in_sample_window = [t for t in all_historical if t["entry_time"] >= start_cutoff]
            leftover_past_data = [t for t in all_historical if t["entry_time"] < start_cutoff]
            
            if len(in_sample_window) < 30:
                continue
                
            # 75% Train / 25% Calibration Split
            split_idx = int(len(in_sample_window) * 0.75)
            train_trades = in_sample_window[:split_idx]
            calib_trades = in_sample_window[split_idx:]
            
            X_train = np.array([t["features"] for t in train_trades])
            y_train = np.array([t["y"] for t in train_trades])
            X_calib = np.array([t["features"] for t in calib_trades])
            
            # Train LightGBM Meta-Model
            clf = lgb.LGBMClassifier(
                n_estimators=100, max_depth=4, learning_rate=0.03,
                subsample=0.8, colsample_bytree=0.8, min_child_samples=15,
                random_state=42, verbosity=-1
            )
            clf.fit(X_train, y_train)
            
            # Calibrate optimal threshold tau* on calibration set
            calib_probs = clf.predict_proba(X_calib)[:, 1]
            for idx, t in enumerate(calib_trades):
                t["prob"] = calib_probs[idx]
                
            thresholds = np.linspace(0.04, 0.45, 60)
            best_tau = 0.10
            best_obj = -999.0
            for tau in thresholds:
                adm = [t for t in calib_trades if t["prob"] >= tau]
                if len(adm) < 6:
                    continue
                tot_r = sum(t["net_r"] for t in adm)
                wr = len([t for t in adm if t["net_r"] > 0]) / len(adm)
                obj = tot_r * wr
                if obj > best_obj:
                    best_obj = obj
                    best_tau = tau
                    
            # ------------------------------------------------------------------
            # 1. In-Sample Backtest Metrics (Across the lookback window)
            # ------------------------------------------------------------------
            all_in_probs = clf.predict_proba(np.array([t["features"] for t in in_sample_window]))[:, 1]
            in_adm = [t for idx, t in enumerate(in_sample_window) if all_in_probs[idx] >= best_tau]
            in_wins = len([t for t in in_adm if t["net_r"] > 0])
            in_wr = (in_wins / len(in_adm) * 100) if in_adm else 0.0
            in_net_r = sum(t["net_r"] for t in in_adm)
            in_avg_r = (in_net_r / len(in_adm)) if in_adm else 0.0
            
            # ------------------------------------------------------------------
            # 2. Forward Live Unseen Evaluation (Oct 21 - Nov 7, 2026)
            # ------------------------------------------------------------------
            live_adm = []
            if live_unseen:
                live_probs = clf.predict_proba(np.array([t["features"] for t in live_unseen]))[:, 1]
                live_adm = [t for idx, t in enumerate(live_unseen) if live_probs[idx] >= best_tau]
                
            live_wins = len([t for t in live_adm if t["net_r"] > 0])
            live_wr = (live_wins / len(live_adm) * 100) if live_adm else 0.0
            live_net_r = sum(t["net_r"] for t in live_adm)
            live_avg_r = (live_net_r / len(live_adm)) if live_adm else 0.0
            
            # ------------------------------------------------------------------
            # 3. Backward Regime Transfer (Stress-test on Leftover Past Data)
            # ------------------------------------------------------------------
            past_adm = []
            if leftover_past_data:
                past_probs = clf.predict_proba(np.array([t["features"] for t in leftover_past_data]))[:, 1]
                past_adm = [t for idx, t in enumerate(leftover_past_data) if past_probs[idx] >= best_tau]
                
            past_wins = len([t for t in past_adm if t["net_r"] > 0])
            past_wr = (past_wins / len(past_adm) * 100) if past_adm else 0.0
            past_net_r = sum(t["net_r"] for t in past_adm)
            past_avg_r = (past_net_r / len(past_adm)) if past_adm else 0.0
            
            results.append({
                "Symbol": sym,
                "Timeframe": timeframe,
                "Horizon": h_label,
                "Years": h_years,
                "InSample_Setups": len(in_sample_window),
                "Tau_Star": round(best_tau, 3),
                "In_Trades": len(in_adm),
                "In_WinRate%": round(in_wr, 1),
                "In_NetR": round(in_net_r, 2),
                "In_AvgR": round(in_avg_r, 3),
                "Live_Trades": len(live_adm),
                "Live_WinRate%": round(live_wr, 1),
                "Live_NetR": round(live_net_r, 2),
                "Live_AvgR": round(live_avg_r, 3),
                "Past_Leftover_Setups": len(leftover_past_data),
                "Past_Trades": len(past_adm),
                "Past_WinRate%": round(past_wr, 1),
                "Past_NetR": round(past_net_r, 2),
                "Past_AvgR": round(past_avg_r, 3),
                "Model_Object": clf,
            })
            
    return pd.DataFrame(results)

# Run benchmark for 1H
df_results_1h = run_lookback_regime_study(timeframe="1h")
# Run benchmark for 4H
df_results_4h = run_lookback_regime_study(timeframe="4h")
print("\n✓ All multi-horizon lookback sweeps completed!")


## Step 4: Visual Scorecard & Comparative Heatmaps

In [ ]:
# 5. Master Comparative Visualizer & Heatmaps

def plot_lookback_scorecard(df_results: pd.DataFrame, timeframe: str = "1h"):
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle(f"Quantitative Lookback Horizon & Regime Transfer Study [{timeframe.upper()}]", fontsize=16, fontweight="bold")
    
    # 1. In-Sample Win Rate vs Lookback Horizon
    sns.lineplot(data=df_results, x="Years", y="In_WinRate%", hue="Symbol", marker="o", ax=axes[0, 0])
    axes[0, 0].set_title("In-Sample Win Rate % vs Lookback Window", fontweight="bold")
    axes[0, 0].set_xlabel("Training Lookback (Years)")
    axes[0, 0].set_ylabel("Win Rate %")
    axes[0, 0].invert_xaxis()
    
    # 2. In-Sample Average Net R per Trade
    sns.barplot(data=df_results, x="Horizon", y="In_AvgR", hue="Symbol", ax=axes[0, 1])
    axes[0, 1].set_title("Average Net R per Trade by Training Horizon", fontweight="bold")
    axes[0, 1].set_xlabel("")
    axes[0, 1].set_ylabel("Avg Net R / Trade")
    axes[0, 1].tick_params(axis='x', rotation=30)
    
    # 3. Live Window Forward Realized Net R (Oct 21 - Nov 7)
    sns.barplot(data=df_results, x="Horizon", y="Live_NetR", hue="Symbol", ax=axes[1, 0])
    axes[1, 0].set_title("Live Unseen Window Realized Net R (Oct 21 - Nov 7, 2026)", fontweight="bold")
    axes[1, 0].set_xlabel("")
    axes[1, 0].set_ylabel("Live Net R")
    axes[1, 0].tick_params(axis='x', rotation=30)
    axes[1, 0].axhline(0, color='black', linestyle='--', alpha=0.6)
    
    # 4. Backward Regime Transfer: Win Rate on Excluded Earlier Past Data
    sns.lineplot(data=df_results[df_results['Past_Trades'] > 0], x="Years", y="Past_WinRate%", hue="Symbol", marker="s", ax=axes[1, 1])
    axes[1, 1].set_title("Backward Regime Transfer: Win Rate on Leftover Past Data (2020-2023)", fontweight="bold")
    axes[1, 1].set_xlabel("Training Lookback (Years)")
    axes[1, 1].set_ylabel("Past Data Win Rate %")
    axes[1, 1].invert_xaxis()
    axes[1, 1].axhline(50, color='red', linestyle=':', alpha=0.7, label='50% Parity')
    
    plt.tight_layout()
    plt.show()

plot_lookback_scorecard(df_results_1h, timeframe="1h")
plot_lookback_scorecard(df_results_4h, timeframe="4h")


## Step 5: Summary Benchmark Tables

In [ ]:
# 6. Concise Scorecard Summary Tables

display_cols = [
    "Symbol", "Horizon", "In_Trades", "In_WinRate%", "In_AvgR", "In_NetR",
    "Live_Trades", "Live_WinRate%", "Live_NetR",
    "Past_Trades", "Past_WinRate%", "Past_NetR"
]

print("=== 1-HOUR HORIZON BENCHMARK SCORECARD ===")
display(df_results_1h[display_cols].sort_values(by=["Symbol", "Horizon"]))

print("\n=== 4-HOUR HORIZON BENCHMARK SCORECARD ===")
display(df_results_4h[display_cols].sort_values(by=["Symbol", "Horizon"]))


## Step 6: Export Best Specialist Models (TypeScript JSON + Zip)

In [ ]:
# 7. Autonomous Best-Model Exporter (TypeScript Compatible JSON + Zip)

def export_best_models(df_results: pd.DataFrame, timeframe: str = "1h"):
    output_dir = Path(f"exported_crypto_models_{timeframe}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    summary = []
    
    for sym in CRYPTOS:
        sub = df_results[df_results["Symbol"] == sym].copy()
        # Find best model balancing In-Sample Net R, Live Net R, and Past Win Rate
        sub["composite_score"] = (sub["In_AvgR"] * 2.0) + (sub["Live_NetR"] * 3.0) + (sub["In_WinRate%"] / 50.0)
        best_row = sub.sort_values(by="composite_score", ascending=False).iloc[0]
        
        clf = best_row["Model_Object"]
        booster = clf.booster_
        model_dict = booster.dump_model()
        
        sym_dir = output_dir / sym
        sym_dir.mkdir(parents=True, exist_ok=True)
        
        with open(sym_dir / "model.json", "w") as f:
            json.dump(model_dict, f, indent=2)
            
        metadata = {
            "model_name": f"distributed_{timeframe}_{sym}_{best_row['Horizon']}",
            "symbol": sym,
            "timeframe": timeframe,
            "selected_lookback": best_row["Horizon"],
            "lookback_years": float(best_row["Years"]),
            "tau_star": float(best_row["Tau_Star"]),
            "in_sample_trades": int(best_row["In_Trades"]),
            "in_sample_win_rate": float(best_row["In_WinRate%"]),
            "in_sample_avg_net_r": float(best_row["In_AvgR"]),
            "live_trades": int(best_row["Live_Trades"]),
            "live_win_rate": float(best_row["Live_WinRate%"]),
            "live_net_r": float(best_row["Live_NetR"]),
            "past_leftover_win_rate": float(best_row["Past_WinRate%"]),
            "num_trees": int(clf.n_estimators),
            "feature_names": FEATURE_NAMES,
        }
        
        with open(sym_dir / "metadata.json", "w") as f:
            json.dump(metadata, f, indent=2)
            
        summary.append(metadata)
        print(f"✓ Exported {sym} [{timeframe}] -> Best Horizon: {best_row['Horizon']} (In WR: {best_row['In_WinRate%']}%, Live Net R: {best_row['Live_NetR']}R)")
        
    # Zip package
    zip_path = f"best_crypto_models_{timeframe}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(output_dir):
            for file in files:
                p = Path(root) / file
                zipf.write(p, p.relative_to(output_dir))
                
    print(f"\n📦 Packaged bundle -> {zip_path}")
    try:
        from google.colab import files
        files.download(zip_path)
        print("🚀 Download triggered in browser!")
    except Exception:
        print("Note: Run inside Google Colab to trigger automatic browser download.")

export_best_models(df_results_1h, timeframe="1h")
export_best_models(df_results_4h, timeframe="4h")
